# Dogs vs. Cats with CNN & Transfer Learning 

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Importing Libraries

In [ ]:
# Standart Kütüphaneler
import os
import math
import zipfile
import warnings
warnings.filterwarnings("ignore")

# Veri İşleme
import numpy as np
import pandas as pd

# Görselleştirme
import matplotlib.pyplot as plt
import seaborn as sns

# Görüntü İşleme
import cv2
from PIL import Image

# Makine Öğrenmesi (sklearn)
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

# TensorFlow / Keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Dense,
    Conv2D,
    Flatten,
    Input,
    MaxPooling2D,
    Dropout,
    BatchNormalization,
    Reshape
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Loading and Organizing Data

In [ ]:
data_dir = '/kaggle/input/competitions/dogs-vs-cats-redux-kernels-edition/'

# Zip dosyalarını çıkartmak için hedef dizin
extract_dir = '/kaggle/working/'

train_zip = os.path.join(data_dir, 'train.zip')
test_zip  = os.path.join(data_dir, 'test.zip')

# Train zip'i çıkart
with zipfile.ZipFile(train_zip, 'r') as zf:
    zf.extractall(extract_dir)

# Test zip'i çıkart
with zipfile.ZipFile(test_zip, 'r') as zf:
    zf.extractall(extract_dir)

In [ ]:
# Güncellenmiş path'ler
train_path = os.path.join(extract_dir, 'train')
test_path  = os.path.join(extract_dir, 'test')

print("Train path:", train_path)
print("Test path:", test_path)
print("Train dosya sayısı:", len(os.listdir(train_path)))
print("Test dosya sayısı:", len(os.listdir(test_path)))

In [ ]:
train_img_list = []
train_label_list = []

# Train klasöründeki tüm dosyaları dolaş
for img_file in os.listdir(train_path):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_full_path = os.path.join(train_path, img_file)
        if os.path.isfile(img_full_path):
            # Dosya adından label'ı çıkar: "cat.0.jpg" -> "cat", "dog.0.jpg" -> "dog"
            label = img_file.split('.')[0]  # 'cat' veya 'dog'
            train_img_list.append(img_full_path)
            train_label_list.append(label)

train_df = pd.DataFrame({'img': train_img_list, 'label': train_label_list})

In [ ]:
train_df.head()

In [ ]:
# CNN için label'ı sayısala çevirelim cat -> 0, dog -> 1
train_df['label_encoded'] = train_df['label'].map({'cat': 0, 'dog': 1})

In [ ]:
train_df.head(10)

In [ ]:
train_df['label'].value_counts()

In [ ]:
test_img_list = []
test_id_list = []

for img_file in os.listdir(test_path):
    if img_file.lower().endswith(('.jpg', '.jpeg', '.png')):
        img_full_path = os.path.join(test_path, img_file)
        if os.path.isfile(img_full_path):
            test_img_list.append(img_full_path)
            test_id_list.append(int(img_file.split('.')[0]))  

test_df = pd.DataFrame({'id': test_id_list, 'img': test_img_list})

In [ ]:
test_df = test_df.sort_values('id').reset_index(drop=True)

In [ ]:
test_df.head(10)

In [ ]:
#Visualition
labels = train_df['label'].unique()  # ['cat', 'dog']
n_samples = 5  # her sınıftan kaç görsel yükleyeceğimizi belirtiyoruz

fig, axes = plt.subplots(2, n_samples, figsize=(15, 6))

for row, label in enumerate(labels):
    img_paths = train_df[train_df['label'] == label]['img'].sample(n_samples, random_state=42)
    
    for col, img_path in enumerate(img_paths):
        image = Image.open(img_path)
        axes[row, col].imshow(image)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(f'{"🐱 Cat" if label == "cat" else "🐶 Dog"}', 
                                      fontsize=13, fontweight='bold', loc='left')

plt.suptitle('Dogs vs Cats - Sample Images', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Normalizing Images

In [ ]:
#  Görüntü Yükleme ve Normalizasyon 
def load_image(img_path, img_size=(128, 128)):    
    img = cv2.imread(str(img_path))

    if img is None:
        return None

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    
    img = cv2.resize(img, img_size)
    img = img.astype("float32") / 255.0           # Normalize [0, 1]

    return img

In [ ]:
#  DataFrame'den Toplu Yükleme Fonksiyonu 
def load_dataset(df, has_label=True, log_step=5000):
    images, labels = [], []

    for idx, row in df.iterrows():
        img = load_image(row['img'])
        if img is not None:
            images.append(img)
            if has_label:
                labels.append(row['label_encoded'])

        if (idx + 1) % log_step == 0:
            print(f"  {idx + 1}/{len(df)} yüklendi.")

    images = np.array(images)
    return (images, np.array(labels)) if has_label else images


print("Train verisi yükleniyor...")
x_train, y_train = load_dataset(train_df, has_label=True)
print(f"✅ Train verisi : {x_train.shape}")   
print(f"✅ Label sayısı : {len(y_train)}")    


print("\nTest verisi yükleniyor...")
x_test = load_dataset(test_df, has_label=False)
print(f"✅ Test verisi  : {x_test.shape}")    

## Train / Validation Split

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train          
)
print(f"Train: {x_train.shape}, Validation: {x_val.shape}")

## CNN Modeling

In [ ]:
model = Sequential([

    Input(shape=(128, 128, 3)),          

    # Block 1
    Conv2D(32, (3,3), padding="same", activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 2
    Conv2D(64, (3,3), padding="same", activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.25),

    # Block 3
    Conv2D(128, (3,3), padding="same", activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    # Block 4 (128x128 için ek blok)
    Conv2D(256, (3,3), padding="same", activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    MaxPooling2D((2,2)),
    Dropout(0.3),

    Flatten(),

    Dense(256, activation='relu', kernel_regularizer=l2(0.001)),
    BatchNormalization(),
    Dropout(0.5),

    Dense(1, activation='sigmoid')       #Binary: 0 cat, 1dog 
])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
print("\nModel Özeti:")
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
]

In [ ]:
# Trainig
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

## Saving the Model

In [ ]:
model.save('dogs_vs_cats_cnn.h5')

In [ ]:
model.save('dogs_vs_cats_cnn.keras')

In [ ]:
# Loss & Accuracy 
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Train Loss',      color='#8502d1')
plt.plot(history.history['val_loss'], label='Validation Loss', color='darkorange')
plt.legend()
plt.title('Loss Evolution')
plt.xlabel('Epoch')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Train Accuracy',      color='#8502d1')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='darkorange')
plt.legend()
plt.title('Accuracy Evolution')
plt.xlabel('Epoch')

plt.tight_layout()
plt.show()

## Predict on the Test

In [ ]:
y_pred_prob     = model.predict(x_test)              
y_pred_classes  = (y_pred_prob > 0.5).astype(int).flatten()  # 0=cat, 1=dog
class_names     = {0: '🐱 Cat', 1: '🐶 Dog'}

# İlk 10 test görselini göster
plt.figure(figsize=(15, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i])
    plt.title(f"{class_names[y_pred_classes[i]]}\n({y_pred_prob[i][0]:.2f})", fontsize=9)
    plt.axis('off')
plt.suptitle('İlk 10 Test Tahmini', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Validation Confusion Matrix
y_val_pred_prob    = model.predict(x_val)
y_val_pred_classes = (y_val_pred_prob > 0.5).astype(int).flatten()

cm_val = confusion_matrix(y_val, y_val_pred_classes)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_val, annot=True, fmt='d', cmap='crest',
            xticklabels=['Cat', 'Dog'],
            yticklabels=['Cat', 'Dog'])
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.title('Confusion Matrix - Validation Seti')
plt.tight_layout()
plt.show()

In [ ]:
submission = pd.DataFrame({
    'id':    test_df['id'],                  
    'label': y_pred_prob.flatten()          
})

submission = submission.sort_values('id').reset_index(drop=True)

In [ ]:
submission.head(15)

In [ ]:
submission.to_csv('submission.csv', index=False)

## Transfer Learning 

In [ ]:
# Parameters
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32

# Data Generators 
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='img',
    y_col='label',                 # cat or dog
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',            
    subset='training',
    shuffle=True
)

val_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    x_col='img',
    y_col='label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',            
    subset='validation',
    shuffle=False                   
)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='img',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode=None,
    shuffle=False
)

print(f"Train   : {train_generator.samples}")
print(f"Val     : {val_generator.samples}")
print(f"Test    : {test_generator.samples}")
print(f"Classes : {train_generator.class_indices}")  # {'cat': 0, 'dog': 1}

In [ ]:
# Transfer Learning - VGG16 
base_model = VGG16(
    weights='imagenet',
    input_shape=(*IMG_SIZE, 3),
    include_top=False
)
base_model.trainable = False       # Önceden eğitilmiş ağırlıkları dondurmak için FALSe diyoruz

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'), 
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]

In [ ]:
history = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
model.save('dogs_vs_cats_TL.h5')

In [ ]:
model.save('dogs_vs_cats_TL.keras')

In [ ]:
# Loss & Accuracy
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Train Loss',      color='#8502d1')
plt.plot(history.history['val_loss'], label='Validation Loss', color='darkorange')
plt.legend()
plt.title('Loss Evolution')
plt.xlabel('Epoch')

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Train Accuracy',      color='#8502d1')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='darkorange')
plt.legend()
plt.title('Accuracy Evolution')
plt.xlabel('Epoch')

plt.tight_layout()
plt.show()